# 00 - Reset: drop the old per-season Bronze schemas

The original Bronze design (one schema per league, one table per season)
hit ~470 tables and triggered a Unity Catalog quota warning (500 tables
per metastore) before Silver or Gold even existed. Dropping all of it to
start over with a leaner design - see docs/schema_notes.md for the entry
logging this decision.

Run the dry-run cell first and check the list/count look right before
running the drop cell - `DROP SCHEMA ... CASCADE` deletes the schema's
managed tables permanently.

In [ ]:
# Dry run - list every bronze_* schema and how many tables it holds
schemas = [row.databaseName for row in spark.sql("SHOW SCHEMAS IN workspace").collect()
           if row.databaseName.startswith("bronze_")]

total_tables = 0
for s in sorted(schemas):
    n = len(spark.sql(f"SHOW TABLES IN workspace.{s}").collect())
    total_tables += n
    print(f"  {s}: {n} table(s)")

print(f"\n{len(schemas)} schemas, {total_tables} tables total - will be PERMANENTLY DROPPED by the next cell")

## Confirm the counts above look right, then run this cell to actually drop everything

In [ ]:
dropped, failed = 0, []

for s in schemas:
    try:
        spark.sql(f"DROP SCHEMA IF EXISTS workspace.{s} CASCADE")
        dropped += 1
    except Exception as e:
        failed.append((s, str(e)))

print(f"Dropped {dropped} schemas, {len(failed)} failed")
for s, err in failed:
    print(f"  FAILED: {s} -> {err[:200]}")